# Entrenamiento Baseline YOLOv8 Multiclase (CRISP-DM Fase 4: Modeling)
### Sistema de Apoyo al Diagnóstico Odontológico en Fotografía Intraoral

Este notebook implementa el pipeline de entrenamiento del modelo baseline para detección de caries dental conforme a las directivas de **reproducibilidad científica**, **idempotencia** y **trazabilidad experimental**:

1. **Alcance Clínico:** Fotografía intraoral de luz visible (vistas frontal, laterales y oclusales; con y sin separadores labiales).
2. **Esquema Multiclase:** 2 clases patológicas (`0: caries_permanente` y `1: caries_decidua`) y controles negativos sanos (0 lesiones).
3. **Fijación de Semillas y Modo Determinista:** Control de pseudoaleatoriedad (`seed = 42`) en Python, NumPy, PyTorch y flags deterministas de Ultralytics YOLO.
4. **Detección Dinámica de Hardware:** Selección automática y transparente de aceleración en Apple Silicon (`mps`), NVIDIA GPU (`cuda:0`) o `cpu`.
5. **Manejo Preventivo de Anotaciones (Clipping):** Aseguramiento estricto de coordenadas normalizadas en $[0.0, 1.0]$ para estabilidad de gradientes.
6. **Estabilidad Numérica en MPS:** Desactivación explícita de half-precision (`half=False`) para prevenir valores `NaN` en pérdidas.
7. **Persistencia de Trazabilidad:** Exportación automática de la ficha técnica de metadatos (`experiment_metadata.json`) para auditoría académica y defensa de tesis.

## 1. Importación de Librerías y Dependencias del Entorno

In [ ]:
import os
import sys
import json
import random
import platform
import shutil
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import yaml
import ultralytics
from ultralytics import YOLO

print(f"Ultralytics version: {ultralytics.__version__}")
print(f"PyTorch version:     {torch.__version__}")

## 2. Fijación de Semillas y Modo Determinista
Garantizamos que todas las operaciones de partición, inicialización de pesos y data augmentation generen resultados matemáticamente idénticos en ejecuciones sucesivas.

In [ ]:
GLOBAL_SEED = 42

def set_deterministic_environment(seed: int = 42) -> None:
    """Fija semillas pseudoaleatorias y activa flags deterministas en todos los backends."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"✓ Entorno determinista configurado exitosamente: GLOBAL_SEED = {seed}")

set_deterministic_environment(GLOBAL_SEED)

## 3. Detección Dinámica y Transparente de Hardware
El pipeline selecciona automáticamente la mejor aceleración disponible sin intervención manual ni alteración de rutas:
- **Apple Silicon (M1 Pro / M2 / M3):** Acelerador `mps` (Metal Performance Shaders).
- **NVIDIA GPU (Google Colab / Servidor Cloud):** Dispositivo `0` con soporte CUDA.
- **CPU:** Fallback universal en caso de no detectarse aceleración por hardware.

In [ ]:
# Detección dinámica y transparente de aceleración por hardware
device = "mps" if torch.backends.mps.is_available() else ("0" if torch.cuda.is_available() else "cpu")

# Identificación diagnóstica de la unidad de cómputo
if device == "mps":
    device_desc = "Apple Silicon Metal Performance Shaders (MPS)"
elif device == "0":
    device_desc = f"NVIDIA CUDA GPU: {torch.cuda.get_device_name(0)} (CUDA v{torch.version.cuda})"
else:
    device_desc = f"CPU Universal ({platform.processor() or platform.machine()})"

print("=" * 75)
print(" REPORTE DE INFRAESTRUCTURA DE HARDWARE Y SOFTWARE")
print("=" * 75)
print(f"  Dispositivo Seleccionado: {device} ({device_desc})")
print(f"  PyTorch Version:          {torch.__version__}")
print(f"  Ultralytics Version:      {ultralytics.__version__}")
print(f"  Python Version:           {platform.python_version()}")
print(f"  Sistema Operativo:        {platform.system()} {platform.release()} ({platform.machine()})")
print("=" * 75)

## 4. Definición de Rutas Portables y Verificación de `data.yaml`
Se resuelven dinámicamente las rutas canónicas del proyecto, garantizando portabilidad absoluta entre plataformas.

In [ ]:
PROJECT_ROOT = Path("..").resolve()
DATA_YAML = PROJECT_ROOT / "ml" / "configs" / "data.yaml"
DATASET_DIR = PROJECT_ROOT / "dataset" / "selected"
RUNS_DIR = PROJECT_ROOT / "ml" / "runs"
MODEL_DIR = PROJECT_ROOT / "ml" / "models"

print(f"Ruta raíz del proyecto: {PROJECT_ROOT.name}")
print(f"Configuración YOLO:     {DATA_YAML.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Directorio dataset:     {DATASET_DIR.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Directorio runs:        {RUNS_DIR.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Directorio modelos:     {MODEL_DIR.relative_to(PROJECT_ROOT).as_posix()}")

# Verificación no destructiva de integridad de data.yaml
assert DATA_YAML.exists(), f"ERROR: Archivo no encontrado: {DATA_YAML}"
with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

assert data_config.get("nc") == 2, f"ERROR: 'nc' debe ser 2, encontrado: {data_config.get('nc')}"
assert data_config.get("names") == ["caries_permanente", "caries_decidua"], f"ERROR: Clases erróneas: {data_config.get('names')}"

resolved_dataset_dir = (DATA_YAML.parent / data_config["path"]).resolve()
assert resolved_dataset_dir.exists(), f"ERROR: Directorio resuelto {resolved_dataset_dir} no existe"

print("\n✓ Verificación de configuración exitosa:")
print(f"  - Clases configuradas (nc=2): {data_config['names']}")
print(f"  - Ruta relativa resuelta:     {resolved_dataset_dir.relative_to(PROJECT_ROOT).as_posix()}")

## 5. Manejo Preventivo de Anotaciones Anómalas (Clipping $[0.0, 1.0]$)
Antes del pase hacia adelante/atrás en PyTorch, auditamos y recortamos preventivamente cualquier caja delimitadora que presente valores fuera del rango normalizado $[0.0, 1.0]$ (resolviendo los 42 casos límite heredados del dataset original de Ahmed et al.).

Esto garantiza **cero advertencias de Ultralytics** y **estabilidad matemática en la función de pérdida CIoU/DFL**.

In [ ]:
def audit_and_clip_dataset_labels(labels_root: Path) -> dict:
    """
    Audita y aplica clipping preventivo a las coordenadas [x_center, y_center, width, height]
    asegurando que todos los valores residan estrictamente en [0.0, 1.0].
    """
    audit = {"total_files": 0, "healthy_controls": 0, "total_boxes": 0, "clipped_boxes": 0}
    
    for split in ["train", "val", "test"]:
        split_dir = labels_root / split
        if not split_dir.exists():
            continue
            
        for lbl_path in split_dir.glob("*.txt"):
            audit["total_files"] += 1
            lines = lbl_path.read_text(encoding="utf-8").splitlines()
            if not lines:
                audit["healthy_controls"] += 1
                continue
                
            clean_lines = []
            file_modified = False
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    xc, yc, w, h = map(float, parts[1:5])
                    audit["total_boxes"] += 1
                    
                    c_xc = max(0.0, min(1.0, xc))
                    c_yc = max(0.0, min(1.0, yc))
                    c_w  = max(0.0, min(1.0, w))
                    c_h  = max(0.0, min(1.0, h))
                    
                    if (c_xc != xc) or (c_yc != yc) or (c_w != w) or (c_h != h):
                        audit["clipped_boxes"] += 1
                        file_modified = True
                        
                    clean_lines.append(f"{cls_id} {c_xc:.6f} {c_yc:.6f} {c_w:.6f} {c_h:.6f}")
                    
            if file_modified:
                lbl_path.write_text("\n".join(clean_lines) + "\n", encoding="utf-8")
                
    return audit

clip_results = audit_and_clip_dataset_labels(DATASET_DIR / "labels")
print(f"Archivos auditados:                    {clip_results['total_files']}")
print(f"Bounding boxes totales auditados:      {clip_results['total_boxes']}")
print(f"Controles sanos certificados:          {clip_results['healthy_controls']}")
print(f"Cajas con clipping preventivo aplicado: {clip_results['clipped_boxes']}")
print("✓ Certificación de normalización: 100% de cajas delimitadoras en rango [0.0, 1.0].")

## 6. Configuración de Hiperparámetros Controlados (M1 Pro / Colab)
Establecemos un conjunto de hiperparámetros estandarizado para la línea base:

| Hiperparámetro | Valor | Justificación Técnica y Clínica |
| :--- | :--- | :--- |
| **Modelo Base** | `yolov8n.pt` | Modelo Nano preentrenado en COCO. Ligero, transferible y óptimo para inferencia de baja latencia. |
| **Épocas (`epochs`)** | `50` | Suficiente para convergencia del baseline sin inducir sobreajuste temprano. |
| **Resolución (`imgsz`)** | `640` | Resolución estándar de YOLO, balance óptimo entre detalle dental y costo de GPU. |
| **Batch Size (`batch`)** | `8` | Adecuado para 16 GB de memoria unificada en Apple Silicon M1 Pro y GPU T4 (16 GB) en Colab. |
| **Precisión (`half`)** | `False` | **Crítico en MPS:** FP16 puede producir gradientes `NaN` en MPS. FP32 asegura estabilidad matemática. |
| **Determinismo** | `deterministic=True`, `seed=42` | Garantía de reproducibilidad científica en la generación de gradientes y augmentation. |
| **Workers** | `2` | Evita saturación de hilos en macOS y previene fugas de memoria en dataloaders. |

In [ ]:
EXPERIMENT_NAME = "caries_yolo_baseline"

# Especificación formal de hiperparámetros de entrenamiento
TRAINING_HYPERPARAMS = {
    "model": "yolov8n.pt",
    "data": str(DATA_YAML),
    "epochs": 50,
    "imgsz": 640,
    "batch": 8,
    "device": device,
    "seed": GLOBAL_SEED,
    "deterministic": True,
    "half": False,           # Obligatorio: previene NaNs en Apple Silicon MPS
    "workers": 2,            # Seguro para Apple Silicon / Windows / Colab
    "project": str(RUNS_DIR),
    "name": EXPERIMENT_NAME,
    "exist_ok": True,
    "plots": True,           # Genera curvas PR, F1, pérdidas y matrices de confusión
    "save": True,
    "save_period": 10,
    "verbose": True,
}

print("Hiperparámetros base formalizados:")
for k, v in TRAINING_HYPERPARAMS.items():
    print(f"  {k:15}: {v}")

## 7. Persistencia de Trazabilidad Científica y Metadatos
Guardamos una ficha técnica exhaustiva en `ml/runs/caries_yolo_baseline/experiment_metadata.json` antes de iniciar el cómputo, garantizando trazabilidad auditable para la defensa de tesis.

In [ ]:
def export_experiment_metadata(config: dict, runs_path: Path, exp_name: str) -> Path:
    """Registra la ficha técnica y de auditoría del experimento en formato JSON."""
    exp_dir = runs_path / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    
    metadata = {
        "experiment_id": exp_name,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "clinical_domain": "e-Health / Intraoral Dental Caries Detection",
        "methodology": "CRISP-DM Phase 4 (Modeling Baseline)",
        "classes": {
            0: "caries_permanente (CPO-D)",
            1: "caries_decidua (ceo-d)"
        },
        "dataset": {
            "config_file": str(DATA_YAML.relative_to(PROJECT_ROOT).as_posix()),
            "total_images": 1957,
            "train_images": 1362,
            "val_images": 290,
            "test_images": 305,
            "unique_patients": 1020,
            "split_ratio": "70/15/15 by patient (zero leakage)",
            "healthy_controls": 33,
        },
        "hardware_environment": {
            "device_flag": config["device"],
            "device_description": device_desc,
            "pytorch_version": torch.__version__,
            "ultralytics_version": ultralytics.__version__,
            "python_version": platform.python_version(),
            "operating_system": f"{platform.system()} {platform.release()} ({platform.machine()})",
        },
        "reproducibility": {
            "seed": config["seed"],
            "deterministic": config["deterministic"],
            "half_precision": config["half"],
        },
        "training_hyperparameters": config,
    }
    
    meta_path = exp_dir / "experiment_metadata.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
        
    print(f"✓ Ficha técnica del experimento guardada en: {meta_path.relative_to(PROJECT_ROOT).as_posix()}")
    return meta_path

metadata_path = export_experiment_metadata(TRAINING_HYPERPARAMS, RUNS_DIR, EXPERIMENT_NAME)

## 8. Carga del Modelo Preentrenado y Ejecución del Entrenamiento
Instanciamos la arquitectura `yolov8n.pt` para transfer learning e invocamos el método `train(**TRAINING_HYPERPARAMS)`.

> **Nota metodológica:** Para ejecutar el entrenamiento completo (50 épocas), descomente la llamada a `model.train()` en la celda a continuación. En entornos sin GPU dedicada o con tiempo limitado, se recomienda exportar este notebook y ejecutarlo en Google Colab con acelerador T4 o en una MacBook Apple Silicon (M1 Pro).

In [ ]:
# Instanciar la arquitectura YOLOv8n con pesos preentrenados
model = YOLO(TRAINING_HYPERPARAMS["model"])

# Ejecución controlada del entrenamiento multiclase determinista
# Descomentar la siguiente línea para iniciar las 50 épocas de entrenamiento:
# results = model.train(**TRAINING_HYPERPARAMS)

## 9. Evaluación del Modelo en el Conjunto de Prueba (`Test Split`)
Una vez concluido el entrenamiento, se evalúan los mejores pesos (`best.pt`) contra el conjunto de prueba independiente (305 imágenes de 153 pacientes reservados sin data leakage), reportando métricas desglosadas por patología.

In [ ]:
BEST_WEIGHTS_PATH = RUNS_DIR / EXPERIMENT_NAME / "weights" / "best.pt"

if BEST_WEIGHTS_PATH.exists():
    print(f"Cargando mejores pesos para evaluación: {BEST_WEIGHTS_PATH.relative_to(PROJECT_ROOT).as_posix()}")
    test_model = YOLO(str(BEST_WEIGHTS_PATH))
    test_results = test_model.val(
        data=str(DATA_YAML),
        split="test",
        project=str(RUNS_DIR),
        name=f"{EXPERIMENT_NAME}_test_evaluation",
        plots=True,
    )
    print("\nMétricas de Rendimiento en Test:")
    print(f"  mAP@50:       {test_results.box.map50:.4f}")
    print(f"  mAP@50-95:    {test_results.box.map:.4f}")
    print(f"  Precision(B): {test_results.box.mp:.4f}")
    print(f"  Recall(B):    {test_results.box.mr:.4f}")
else:
    print(f"[Info] Pesos '{BEST_WEIGHTS_PATH.name}' pendientes de entrenamiento en {RUNS_DIR.name}/{EXPERIMENT_NAME}.")

## 10. Exportación de Pesos al Pipeline de Inferencia (`ml/models/best.pt`)
Para desacoplar el entorno de experimentación (`ml/runs/`) del pipeline de servicio en producción (`backend/app/infrastructure/ml/worker.py`), copiamos los pesos óptimos a `ml/models/best.pt`.

In [ ]:
EXPORTED_MODEL_PATH = MODEL_DIR / "best.pt"

if BEST_WEIGHTS_PATH.exists():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BEST_WEIGHTS_PATH, EXPORTED_MODEL_PATH)
    print(f"✓ Pesos exportados exitosamente a: {EXPORTED_MODEL_PATH.relative_to(PROJECT_ROOT).as_posix()}")
    print(f"  Tamaño del modelo: {EXPORTED_MODEL_PATH.stat().st_size / (1024 * 1024):.2f} MB")
else:
    print(f"[Info] La exportación a '{EXPORTED_MODEL_PATH.name}' se completará una vez entrenado el modelo.")